![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 42 -- Lab 1: First Contact

A radio observatory records thousands of candidate signals from space, and experts rate each one: how likely is it a real attempt at contact? You write the program that gives every new recording its contact probability, a number from 0 to 100. Your file is scored with MSE, the mean of the squared differences between your numbers and the experts' numbers: lower is better.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Radio recordings of 50 numbers each (`band_00` to `band_49`). The training recordings carry the experts' contact probability (`contact`); the test recordings do not, and only the scorer holds their answers. The setup cell leaves in `DATA_DIR`: `train.csv` (`id`, the 50 bands, `contact`), `test.csv` (`id` and the 50 bands) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="uns-03", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("libraries ready")


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/first-contact")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The task

**The story.** Radio telescopes record signals from deep space. Experts rate each recording: is it a natural
source, or someone trying to make contact? Rating by hand is slow, so the observatory wants a program that rates
every new recording.

**One data point** is one recording: an `id` and 50 numbers, `band_00` to `band_49`. The organisers say the numbers
describe the recording (how its frequency shifts, how loud it is, how fast it fades) but not which number is which:
treat them as 50 unnamed measurements.

**The answer** is `contact`, a number from 0 to 100 given by the experts. 0 means surely a natural source, such as
a pulsar (a star that sends regular radio pulses); 100 means surely an attempt at contact.

**The files.** `train.csv` holds recordings with their `contact`; `test.csv` holds the recordings you answer for,
without it. The first two rows of `train.csv` (3 of the 50 bands shown):

| id | band_00 | band_01 | band_02 | ... | band_49 | contact |
|---|---|---|---|---|---|---|
| 0 | 1.74 | 7.59 | 5.19 | ... | -2.07 | 83.42 |
| 1 | 5.74 | 5.27 | 3.52 | ... | 6.42 | 79.24 |

**The organisers' warning.** *Some recordings are fake signals: bursts of random radiation that look like a signal
but have no structure. Real transmissions do not use all 50 dimensions: they lie on a few surfaces of lower dimension
inside the 50.* In plain words: the 50 numbers of a real transmission are not free; they follow a pattern that a few
numbers can describe.

**The file you hand in**, `submission.csv`, has one row per test recording, in the format of
`sample_submission.csv`:

| id | contact |
|---|---|
| 4000 | 50.0 |
| 4001 | 50.0 |

**The score** is MSE (mean squared error) on the test recordings, computed by the scorer cell in Part 2. Lower is
better; 0 would be perfect.

**The rules.** Any library; no outside data. Your notebook must run from start to end.

**The starter** (next cell) fits one linear regression on the 50 bands of every training recording. The score of
its file is the **baseline**; your points count from there.

**Today you work alone.** Nobody helps with the code and AI assistants are off. This morning's slides are yours
to use.


---
# The starter solution

Every olympiad task comes with a starter: something that hands in a valid file. Its score is the **baseline**, and your points count up from it. This task came only with a sample file of random numbers, so the starter below was written for this lab: the simplest model, trained on every recording. Read it before you begin; you may reuse any of its lines.

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# The 50 bands are the inputs; contact is the answer
bands = [col for col in train.columns if col.startswith("band_")]
X_train = train[bands].values
y_train = train["contact"].values
X_test = test[bands].values

# One linear regression on every training recording
model = LinearRegression()
model.fit(X_train, y_train)

submission = pd.DataFrame({"id": test["id"], "contact": model.predict(X_test)})
submission.to_csv("submission.csv", index=False)
print(f"Submission file saved: {len(submission)} rows")
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the data before you build anything.

## Task 1: Load the files

Read the three CSV files of `DATA_DIR` into `train`, `test` and `sample`. Print the shape of each.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample)


## Task 2: Look at the training recordings

Store the first five rows of `train` in `first_rows` and its number of rows in `n_rows`. Print `first_rows`, then the smallest, the mean and the largest `contact` of `train`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task. Run the cell; you will use `score` all afternoon.

In [ ]:
from sklearn.metrics import mean_squared_error


def score(answers, numbers):
    """The score of a file: the mean squared error (MSE) of its numbers against the answers. Lower is better."""
    return float(mean_squared_error(answers, numbers))


print(round(score([10, 50, 90], [20, 50, 80]), 2))


## Task 3: What the scorer gives

The scorer cell above prints 66.67 for three recordings. Which sentence says what `score` gives?

- **a** The mean distance between each number and its answer, so the numbers above are off by about 67 on average.
- **b** The mean of the squared differences between each number and its answer: (10 x 10 + 0 + 10 x 10) / 3 = 66.67.
- **c** The share of numbers that land within 10 of their answer, times 100: two of the three, so 66.67.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two files that need no model score? `fifty` gives every recording the number 50; `mean` gives every recording the mean `contact` of `train`. Score both on the recordings of `train` against their `contact` with `score`, and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
# Your code here


## Task 5: What the metric rewards

File A misses every recording by 2. File B is exactly right on nine recordings in ten, but misses the tenth by 20. Which file has the lower (better) score?

- **a** File B, because it is exactly right on nine recordings in ten, while file A is right on none of them.
- **b** File A: it pays 2 x 2 = 4 on each recording, while B pays 20 x 20 = 400 on one in ten, so 40 in the mean.
- **c** Neither: both files miss by 2 per recording on average, so the two files get the same score.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, `id_col`, the name of its id column, and `pred_col`, the name of the column of your numbers. Read all three from what you loaded; do not type them.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: aim to hand it in within the first hour, then make it better.

## Task 8: Hold out validation recordings

Split `train` into a part `tr` and a validation part `va` of one recording in five, with `random_state=0`. Keep the ids of the two parts in `train_ids` and `val_ids`. Print the number of recordings in each part.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


## Task 9: The numbers the model reads

Build `X_tr`, the table of numbers your model reads for the recordings it learns from (one row per recording), and `y_tr`, their `contact`. Build `X_va` and `y_va` the same way for the recordings you validate on. Print the shapes.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(X_tr, y_tr)


## Task 10: A first model

Train a first model in `model` on `X_tr` and `y_tr`. It must give one number per recording. Print what it gives the first three rows of `X_va`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, X_va)


## Task 11: Score the validation recordings

Store in `val_score` the `score` of your model's numbers for the rows of `X_va` against `y_va`: one number. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="mse")


## Task 12: Write the submission file

Write `submission.csv` in the format of `sample`: for every recording of `test`, its `id` and your number. Leave the index out. Print its shape and its first rows.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Debugging drill

A classmate tried to list the five test recordings with the highest numbers of your file. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it prints `drill_top`, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-dat-1")


## Task 13: Repair the classmate's cell

Change as little as you can: the table and the five rows stay.

In [ ]:
ranked = submission.sort_values(by="contact_probability", ascending=False)
drill_top = ranked.head(5).values
print(drill_top)


In [ ]:
ck.drill_check("bug-dat-1", drill_top)


---
# Part 5 -- Challenge: your best file

Your first file is on the board. Now make it better, one change at a time. A change worth keeping comes from something you found in the data or in the rules, and it lowers the score of the validation recordings.

## Task 14: Look again

Look again at the data, all of it, for something your first file does not use yet. Print what you find.

In [ ]:
# Your code here


## Task 15: Try a better way

Make at least one change to your pipeline. Score it on your validation recordings with `score`, and print it next to `val_score`.

In [ ]:
# Your code here


## Task 16: Hand in your best file

Write `submission.csv` again with your best way. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
## Task 17: Reflect

Which change moved your validation score most, and how did you find it: in the data, in the rules or by trying? What did you look at that the starter never looked at? If you met this task again with one hour, what would you do in the first 15 minutes?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task first**: the scorer read as code, two files without a model, what MSE rewards, and the format of the file, all before any model.
- **A first honest file**: a simple model, scored on validation recordings with the task's own metric, handed in early.
- **Look before you trust**: every row of the data looked at, counted and drawn, and what was found used.
- **Keep a change only when the score says so**: each change scored on validation recordings with the task's own metric before a new file.
- **Debugging**: a missing column read from the error message, located and repaired against the clock.

---

Made By **Sattam Altwaim**